In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)  # Seed for reproducibility

D1 = np.array([[0,0], [0,0]])

from abMPH import compute_absolute_MPH0_multi_critical_filtration, compute_abs_mph0
from scipy.spatial import distance
# compute the absolute MPH0 with multiple critical values
n = D1.shape[0]
pd_mat = distance.squareform(distance.pdist(D1)) 
betti0s, betti1s, betti2s = compute_absolute_MPH0_multi_critical_filtration(pd_mat)
print("betti0s: ", betti0s)
print("betti1s: ", betti1s)
print("betti2s: ", betti2s)



betti0s:  [[ 0.  0.]
 [ 0. -1.]
 [ 0.  0.]
 [ 0. -1.]]
betti1s:  [[ 0.  0.]
 [ 0.  0.]
 [ 0. -1.]]
betti2s:  []


In [2]:
n = 2
vertices = [0, 1]
edges = [[0, 1]]
filt_func_v = np.array([[.0, 0], [0, 0]])
filt_func_e = np.array([[0.0, 0]])

mph_res = compute_abs_mph0(vertices, edges, filt_func_v, filt_func_e)
mph_res

{'b_0': [(0.0, 0.0)], 'b_1': [], 'b_2': [], 'b_0_1': [], 'M': []}

In [3]:
vertices = [0, 1, 2, 3]
edges = [[0, 1], [2, 3], [0,2], [1, 3]]
filt_func_v = np.array([[.0, 0],  # u_0 
                        [.0, -1], # u_1
                        [.0, 0],  # v_0
                        [.0, -1]  # v_1
                        ])
filt_func_e = np.array([[0.0, 0], # e_u
                        [0.0, 0], # e_v
                        [0.0, 0], # e_0
                        [0.0, -1]  # e_1
                        ])

mph_res = compute_abs_mph0(vertices, edges, filt_func_v, filt_func_e)
mph_res

{'b_0': [(0.0, -1.0)], 'b_1': [], 'b_2': [], 'b_0_1': [(0.0, 0.0)], 'M': []}

In [2]:
# Point cloud to 1-critical filtration
def point_cloud_to_1_critical_filtration(points, x_y_swapped=False):
    """
    Convert a point cloud to a 1-critical filtration.
    Input:
        points: a numpy array of shape (n, d)
    Output:
        vertices: a list of vertices (int)
        edges: a list of edges, each edge is a tuple of two integers
        filt_func_v: a numpy array of shape (n, 2) filtration values for vertices
        filt_func_e: a numpy array of shape (n, 2) filtration values for edges
    """
    # Compute the distance matrix
    D = distance.squareform(distance.pdist(points)) 
    n = D.shape[0]

    # create the degree map
    degrees = np.arange(0, n)
    
    # Create a list of vertices 
    # (n*n corresponding to the pairwise distance matrix, i//n is the index of the point)
    # we consider row-wise indexing
    vertices = np.arange(n*n)

    # For each pair of points, get the sorted rows of the distance matrix
    sorted_edge_lengths = np.sort(D, axis=1)

    # Assign filtration values for the vertices
    filt_func_verties = []
    for idx in range(n):
        # concatenate sorted_edge_lengths[idx] with [0, -1, -2, -3, ...] to form a 2D array,
        # it will be used as the filtration values for the vertices
        filt_func_verties.append(np.column_stack((sorted_edge_lengths[idx], -degrees)))
    filt_func_verties = np.vstack(filt_func_verties)

    # Add edges purely on the vertices
    edges = []
    for idx in range(n):
        for jdx in range(n-1):
            # each row has n-1 edges
            row_start_index = idx*n
            edges.append([row_start_index + jdx, row_start_index + jdx + 1 ])
    edges = np.vstack(edges)

    # Add corresponding egde filtration values
    inds = - np.arange(0, n)
    filt_func_edges = [np.column_stack([sorted_edge_lengths[i][1:], inds[:-1]]) for i in range(n)]
    filt_func_edges = np.vstack(filt_func_edges)
    
    
    # Loop over n choose pairs of vertices to add "true" edges 
    pair_edges = [] # edges from pair of vertices 
    filt_func_edges_pair_vertices = [] # filtration values on them
    for i in range(n):
        for j in range(i+1,n):
            
            # take maximum of edges lengths of v[i] and v[j]
            max_rs = np.max([sorted_edge_lengths[i], sorted_edge_lengths[j]], axis= 0)

            # the edge appears at length that is greater than or equal to the distance d(v_i, v_j)
            for index, r in enumerate(max_rs):
                if r >= D[i,j]:
                    # Add the edge filtration value
                    filt_func_edges_pair_vertices.append([max_rs[index], -degrees[index]])

                    # determine the indices of its two end points in pairwise distance matrix
                    i_PD_idx, j_PD_idx = i*n + index, j*n + index

                    # Add the edge
                    pair_edges.append([i_PD_idx, j_PD_idx])
    
    # Append those edges to above 
    pair_edges = np.vstack(pair_edges)
    filt_func_edges_pair_vertices = np.vstack(filt_func_edges_pair_vertices)
    
    edges = np.vstack([edges, pair_edges])
    filt_func_edges = np.vstack([filt_func_edges, filt_func_edges_pair_vertices])

    # swap the two columns for filtration function of the vertices and edges
    if x_y_swapped:
        filt_func_edges = filt_func_edges[:, [1, 0]]
        filt_func_verties = filt_func_verties[:, [1, 0]]

    return vertices, edges, filt_func_verties, filt_func_edges



In [5]:
pts = np.array([
    [0,0],
    [3,4],
    [-5,12]
    ])
# pts = np.array([[0,0], [0,0]])
Vs, Es, F_Vs, F_Es = point_cloud_to_1_critical_filtration(pts)
# Vs, Es, F_Vs, F_Es
mph_res = compute_abs_mph0(Vs, Es, F_Vs, F_Es)
mph_res

{'b_0': [(0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (5.0, -1.0),
  (11.313708498984761, -2.0)],
 'b_1': [(5.0, 0.0),
  (5.0, 0.0),
  (11.313708498984761, -1.0),
  (11.313708498984761, 0.0)],
 'b_2': [],
 'b_0_1': [(13.0, -2.0), (13.0, -1.0), (13.0, -1.0)],
 'M': [(2, 1, -1),
  (4, 1, 1),
  (3, 2, -1),
  (4, 2, 1),
  (4, 3, -1),
  (5, 3, 1),
  (1, 4, -1),
  (4, 4, 1)]}

All good now!!!

In [5]:
import time
from abMPH import compute_absolute_MPH0_multi_critical_filtration, compute_abs_mph0
from scipy.spatial import distance
import numpy as np
# Load points from text file
pts = np.loadtxt('/home/yluo/Documents/rivet-python/example/random_points.txt', delimiter=',', skiprows=7)  # skiprows=1 ignores header

# Start timing
t0 = time.time()

# start compute
Vs, Es, F_Vs, F_Es = point_cloud_to_1_critical_filtration(pts)

t1 = time.time()
mph_res = compute_abs_mph0(Vs, Es, F_Vs, F_Es)

# End timing
t2 = time.time()

# Calculate and print execution time
construct_fil_time = t1 - t0
execution_time = t2 - t1
print(f"construction time: {construct_fil_time:4f} seconds")
print(f"Execution time: {execution_time:.4f} seconds")

construction time: 0.653848 seconds
Execution time: 87.7193 seconds


In [4]:
mph_res

{'b_0': [(0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),
  (0.0, 0.0),